# To'liq fine-tune (Full Fine-Tune)

LoRA'dan farqi: **hamma vaznlar** o'qitiladi (adapter yo'q). Shuning uchun:
- **LR ancha past** — `2e-5` (LoRA'dagi `1e-4` EMAS; yuqori LR to'liq FT'ni buzadi)
- **gradient checkpointing** — 0.5B to'liq FT + Adam optimizer holatlari og'ir, xotira tejaymiz
- **catastrophic forgetting xavfi** — model o'zbekchani o'rganib, inglizcha/ruschani unutishi mumkin.
  Oxirgi katak aynan shuni tekshiradi. Xavfni kamaytirish uchun: past LR, kam epoch, early stopping.

Loss LoRA notebook'dagi bilan bir xil — oxirgi token `<|im_end|>` ehtimoli ustida (EOU semantikasi).


In [1]:
from huggingface_hub import snapshot_download
path = snapshot_download("livekit/turn-detector", revision="v0.4.1-intl")
print(path)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

/Users/rakhmonberdiev/.cache/huggingface/hub/models--livekit--turn-detector/snapshots/87e35fcb1e60a569bea70346191c4886ea92e281


In [2]:
import json, time, unicodedata, re
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

HG_MODEL = "livekit/turn-detector"
ONNX_FILENAME = "model_q8.onnx"
MODEL_REVISION = "v0.4.1-intl"
MAX_HISTORY_TOKENS = 128
TEST_LANGUAGE = "en"

onnx_path = hf_hub_download(HG_MODEL, ONNX_FILENAME, subfolder="onnx",
                           revision=MODEL_REVISION, local_files_only=True)
lang_path = hf_hub_download(HG_MODEL, "languages.json",
                           revision=MODEL_REVISION, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(HG_MODEL, revision=MODEL_REVISION,
                                          local_files_only=True, truncation_side="left")
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

with open(lang_path) as f:
    languages = json.load(f)
threshold = languages.get(TEST_LANGUAGE, {}).get("threshold", 0.5)

# training uchun kerak bo'ladiganlar
tok = tokenizer
IM_END_ID = tok.convert_tokens_to_ids("<|im_end|>")

def build_text(messages):
    convo = tok.apply_chat_template(messages, add_generation_prompt=False,
                                    add_special_tokens=False, tokenize=False)
    return convo[:convo.rfind("<|im_end|>")]

print(f"Ready. threshold={threshold:.4f}, im_end_id={IM_END_ID}")


Ready. threshold=0.0110, im_end_id=151645


In [3]:
import json, torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

MODEL, REV, MAX_LEN = "livekit/turn-detector", "v0.4.1-intl", 128

class EOUDataset(Dataset):
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        enc = tok(build_text(r["messages"]), add_special_tokens=False,
                  truncation=True, max_length=MAX_LEN)
        return {"input_ids": enc["input_ids"], "label": float(r["label"])}

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    ids_out, attn, labels = [], [], []
    for b in batch:
        ids = b["input_ids"]; pad = maxlen - len(ids)
        ids_out.append([tok.pad_token_id] * pad + ids)   # LEFT padding — muhim
        attn.append([0] * pad + [1] * len(ids))
        labels.append(b["label"])
    return {"input_ids": torch.tensor(ids_out),
            "attention_mask": torch.tensor(attn),
            "labels": torch.tensor(labels)}

class EOUTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        logprobs = torch.log_softmax(out.logits[:, -1, :].float(), dim=-1)
        p = logprobs[:, IM_END_ID].exp().clamp(1e-6, 1 - 1e-6)
        loss = -(labels * p.log() + (1 - labels) * (1 - p).log()).mean()
        return (loss, out) if return_outputs else loss


In [4]:
# --- TO'LIQ FINE-TUNE: LoRA yo'q, hamma vaznlar o'qitiladi ---
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, dtype=torch.float32)

# xotira tejash (0.5B to'liq FT + Adam holatlari og'ir):
model.gradient_checkpointing_enable()
model.config.use_cache = False          # gradient checkpointing bilan mos

n_all = sum(p.numel() for p in model.parameters())
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_tr:,} / {n_all:,}  ({100*n_tr/n_all:.1f}%)")   # 100%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 258,692,992 / 258,692,992  (100.0%)


In [ ]:
from transformers import EarlyStoppingCallback

args = TrainingArguments(
    output_dir="./uz_eou_full/v1_Num",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,       # effektiv batch = 16
    per_device_eval_batch_size=16,
    num_train_epochs=3,                  # to'liq FT: 3 yetarli (5 -> overfit/forgetting xavfi)
    learning_rate=1e-5,                  # TO'LIQ FT PAST LR (LoRA'dagi 1e-4 EMAS!)
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

trainer = EOUTrainer(
    model=model, args=args,
    train_dataset=EOUDataset("./uz_dataset/uz_eou_train_num.jsonl"),
    eval_dataset=EOUDataset("./uz_dataset/uz_eou_val_num.jsonl"),
    data_collator=collate,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


/Users/rakhmonberdiev/Research/TurnDetecting/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.026939,0.022748
2,0.000231,0.004845
3,0.000004,0.001499


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/rakhmonberdiev/Research/TurnDetecting/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/rakhmonberdiev/Research/TurnDetecting/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=2190, training_loss=0.15279085239253037, metrics={'train_runtime': 3552.0828, 'train_samples_per_second': 9.864, 'train_steps_per_second': 0.617, 'total_flos': 2850874800280320.0, 'train_loss': 0.15279085239253037, 'epoch': 3.0})

In [6]:
import numpy as np
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def predict_pt(messages):
    enc = tok(build_text(messages), add_special_tokens=False, return_tensors="pt",
              truncation=True, max_length=128)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    lp = torch.log_softmax(model(**enc).logits[0, -1, :].float(), dim=-1)
    return lp[IM_END_ID].exp().item()

model.eval()
rows = [json.loads(l) for l in open("./uz_dataset/uz_eou_val.jsonl", encoding="utf-8")]
probs = np.array([predict_pt(r["messages"]) for r in rows])
labels = np.array([r["label"] for r in rows])
print("val AUC:", roc_auc_score(labels, probs))

# eng yaxshi threshold (bank uchun TNR muhimroq — chala gapni kesib yubormaslik)
for th in np.logspace(-4, -0.3, 30):
    tpr = ((probs >= th) & (labels == 1)).sum() / max((labels == 1).sum(), 1)
    tnr = ((probs <  th) & (labels == 0)).sum() / max((labels == 0).sum(), 1)
    print(f"{th:.5f}  tpr={tpr:.3f}  tnr={tnr:.3f}  bal={(tpr+tnr)/2:.3f}")


val AUC: 0.8633411494996385
0.00010  tpr=0.856  tnr=0.657  bal=0.757
0.00013  tpr=0.852  tnr=0.660  bal=0.756
0.00018  tpr=0.844  tnr=0.669  bal=0.757
0.00024  tpr=0.839  tnr=0.672  bal=0.756
0.00032  tpr=0.833  tnr=0.681  bal=0.757
0.00043  tpr=0.828  tnr=0.689  bal=0.759
0.00058  tpr=0.825  tnr=0.701  bal=0.763
0.00078  tpr=0.824  tnr=0.707  bal=0.766
0.00105  tpr=0.817  tnr=0.712  bal=0.765
0.00141  tpr=0.814  tnr=0.716  bal=0.765
0.00189  tpr=0.808  tnr=0.723  bal=0.765
0.00253  tpr=0.803  tnr=0.730  bal=0.767
0.00340  tpr=0.802  tnr=0.732  bal=0.767
0.00456  tpr=0.796  tnr=0.738  bal=0.767
0.00611  tpr=0.796  tnr=0.745  bal=0.771
0.00820  tpr=0.788  tnr=0.750  bal=0.769
0.01100  tpr=0.788  tnr=0.753  bal=0.770
0.01476  tpr=0.782  tnr=0.756  bal=0.769
0.01979  tpr=0.777  tnr=0.762  bal=0.770
0.02655  tpr=0.775  tnr=0.767  bal=0.771
0.03562  tpr=0.768  tnr=0.774  bal=0.771
0.04779  tpr=0.766  tnr=0.777  bal=0.772
0.06411  tpr=0.760  tnr=0.782  bal=0.771
0.08600  tpr=0.757  tnr=0.787

In [7]:
# --- Catastrophic forgetting tekshiruvi (to'liq FT eng katta xavf) ---
# Inglizcha hali ishlayaptimi? (1=tugagan, 0=tugallanmagan)
print("INGLIZCHA (unutilmadimi?)")
for t, exp in [("what time is it", 1),
               ("so basically my flight got cancelled and", 0),
               ("the thing is", 0),
               ("i need to think about that for a moment", 1)]:
    p = predict_pt([{"role": "user", "content": t}])
    print(f"  {'OK' if (p>=0.05)==(exp==1) else 'XX'} {p:.4f} (kut {exp})  {t}")

print("\nO'ZBEKCHA raqam chegaralari")
for msgs, exp in [
    ([{"role":"assistant","content":"aloqa uchun telefon raqamingiz qanday"},
      {"role":"user","content":"to'qson birlik yetti besh uch to'rt olti bir ikki"}], 1),   # 9
    ([{"role":"assistant","content":"aloqa uchun telefon raqamingiz qanday"},
      {"role":"user","content":"to'qson birlik yetti besh uch to'rt olti bir"}], 0),         # 8
    ([{"role":"assistant","content":"tug'ilgan yilingizni ayting"},
      {"role":"user","content":"bir ming to'qqiz yuz to'qson uchinchi yil"}], 1),
    ([{"role":"assistant","content":"tug'ilgan sanangizni kun oy va yil ko'rinishida to'liq ayting"},
      {"role":"user","content":"bir ming to'qqiz yuz to'qson uchinchi yil"}], 0),
]:
    p = predict_pt(msgs)
    print(f"  {'OK' if (p>=0.05)==(exp==1) else 'XX'} {p:.4f} (kut {exp})  {msgs[-1]['content']}")


INGLIZCHA (unutilmadimi?)
  OK 0.0988 (kut 1)  what time is it
  OK 0.0001 (kut 0)  so basically my flight got cancelled and
  OK 0.0008 (kut 0)  the thing is
  OK 0.5939 (kut 1)  i need to think about that for a moment

O'ZBEKCHA raqam chegaralari
  OK 0.9999 (kut 1)  to'qson birlik yetti besh uch to'rt olti bir ikki
  OK 0.0336 (kut 0)  to'qson birlik yetti besh uch to'rt olti bir
  OK 1.0000 (kut 1)  bir ming to'qqiz yuz to'qson uchinchi yil
  OK 0.0000 (kut 0)  bir ming to'qqiz yuz to'qson uchinchi yil


## ONNX eksport (deploy uchun)

Plagin modelni ONNX (int8) formatida ishlatadi. To'liq FT tugagach:

```bash
pip install optimum[exporters]
optimum-cli export onnx --model ./uz_eou_full/final ./uz_eou_full/onnx --task text-generation
# so'ng int8 kvantlash (onnxruntime.quantization.quantize_dynamic)
```

LoRA'dan farqi: bu yerda `merge_and_unload` KERAK EMAS — model allaqachon to'liq, to'g'ridan-to'g'ri eksport qilinadi.
